##### Version 5.5:

On récupere les différents coefficients à partir d'une fiche exel via openpyxl, on calcul donc les moyennes sur les des semestres allant du premier au dernier.

In [1]:
from tkinter import *
from tkinter import ttk
import openpyxl
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import (FigureCanvasTkAgg, NavigationToolbar2Tk)

fexel=openpyxl.load_workbook('coefficients.xlsx', read_only = True)
fenetre = Tk()
fenetre.title("Saisie des notes")

frm = ttk.Frame(fenetre, padding=10)
frm.grid()

res = [
    "R101", "R102", "R103", "R104", "R105", "R106", "R107", "R108", "R109", "R110",
    "R111", "R112", "R113", "R114", "R115", "SAE11", "SAE12", "SAE13", "SAE14", "SAE15", "SAE16"
]

entries = []
# Obtention des coefficients depuis un fichier exel
def coef_extractor(feuille):
    lst_coef_ue1=[]
    lst_coef_ue2=[]
    lst_coef_ue3=[]
    lst_coef_ue4=[]
    lst_coef_ue5=[]
    rtn=[]
    for row in feuille.iter_rows(min_row=3, min_col=1):
        for cell in row[1:]:
            if cell.value is None:
                cell.value=0
            if cell.column_letter == "B":
                lst_coef_ue1.append(int(cell.value))
            if cell.column_letter == "C":
                lst_coef_ue2.append(int(cell.value))
            if cell.column_letter == "D":
                lst_coef_ue3.append(int(cell.value))
            if cell.column_letter == "E":
                lst_coef_ue4.append(int(cell.value))
            if cell.column_letter == "F":
                lst_coef_ue5.append(int(cell.value))
    if (sum(lst_coef_ue4)+sum(lst_coef_ue5)) ==0:
        rtn.append(lst_coef_ue1)
        rtn.append(lst_coef_ue2)
        rtn.append(lst_coef_ue3)
    else:
        rtn.append(lst_coef_ue1)
        rtn.append(lst_coef_ue2)
        rtn.append(lst_coef_ue3)
        rtn.append(lst_coef_ue4)
        rtn.append(lst_coef_ue5)
    return rtn
    
# Création de l'interface graphique et de la liste des entrées
for n, name in enumerate(res):
    txt = ttk.Label(frm, text=name)
    txt.grid(column=0, row=n, padx=5, pady=2)

    entry = ttk.Entry(frm)
    entry.grid(column=1, row=n, padx=5, pady=2)
    entries.append(entry)
n+=1
txt = ttk.Label(frm, text="Semestre")
txt.grid(column=0, row=n, padx=5, pady=2)

entry = ttk.Entry(frm)
entry.grid(column=1, row=n, padx=5, pady=2)
entries.append(entry)
    
# Calcul de moyenne
def calcul(lst_notes, coefficients):
    total = 0
    total_coef = 0
    for coef, notes in zip(coefficients, lst_notes):
        total += notes * coef
        total_coef += coef
    if total_coef == 0:
        return 0.0
    return total / total_coef

# Résultat du jury
def jury(moyennes):
    nb_ue_sous_10 = 0
    nb_ue_sous_8 = 0
    for moyenne in moyennes:
        if moyenne < 10:
            nb_ue_sous_10 += 1
        if moyenne < 8:
            nb_ue_sous_8 += 1
    if nb_ue_sous_10 == 0:
        print("Vous avez au moins 10 dans toutes les UE. Félicitations, vous validez votre année !")
    elif nb_ue_sous_8 > 0:
        print("Une UE est sous 8, vous n'avez donc pas validé votre année.")
    elif nb_ue_sous_10 == 1:
        print("Une de vos UE est sous 10, vous devrez valider vos UE en deuxième année.")
    else:
        print("Vous avez plus de 2 notes en dessous de 10, vous ne validez pas votre année.")

# Fonction pour créer et afficher le graphique dans une nouvelle fenêtre ou un cadre
def plot_graph(moyennes, bol):
    plot_window = Toplevel(fenetre)
    plot_window.title("Graphique des moyennes")

    fig = Figure(figsize=(5, 4), dpi=100)
    ax = fig.add_subplot(111)
    if bol==0:
        ues = ["UE1", "UE2", "UE3"]
        ax.bar(ues, moyennes, color=["skyblue", "orange", "green"])
        ax.set_title("Moyennes des trois UE")
    else:
        ues = ["UE1", "UE2", "UE3", "UE4", "UE5"]
        ax.bar(ues, moyennes, color=["skyblue", "orange", "green", "red", "blue"])
        ax.set_title("Moyennes des cinque UE")
    ax.set_xlabel("Unités d'enseignement")
    ax.set_ylabel("Moyenne sur 20")
    ax.set_ylim(0, 20)
    
    for i, moyenne in enumerate(moyennes):
        ax.text(i, moyenne + 0.2, f"{moyenne:.2f}", ha="center")

    canvas = FigureCanvasTkAgg(fig, master=plot_window)
    canvas.draw()
    canvas.get_tk_widget().pack(side=TOP, fill=BOTH, expand=1)

    toolbar = NavigationToolbar2Tk(canvas, plot_window)
    toolbar.update()
    canvas.get_tk_widget().pack(side=TOP, fill=BOTH, expand=1)

# Récupération des données transmises par les entrées
def valider():
    lst_notes = []
    bol=0
    for entry in entries[:-1]:
        val = entry.get()
        val_float = float(val)
        if val_float < 0.0:
            lst_notes.append(0.0)
        else:
            lst_notes.append(val_float)
    # Obtention des coefficients selon les semestres
    lst_coef = coef_extractor(fexel[entries[-1].get()])
    coefficients_ue1=lst_coef[0]
    coefficients_ue2=lst_coef[1]
    coefficients_ue3=lst_coef[2]
    print(entries[-1].get(), type (entries[-1].get()))
    if not (entries[-1].get() == "semestre1" or entries[-1].get() == "semestre2"):
        coefficients_ue4=lst_coef[3]
        coefficients_ue5=lst_coef[4]
        bol = 1
    
    # Calcul des moyennes
    moyenne_ue1 = calcul(lst_notes, coefficients_ue1)
    moyenne_ue2 = calcul(lst_notes, coefficients_ue2)
    moyenne_ue3 = calcul(lst_notes, coefficients_ue3)
    if not (entries[-1].get() == "semestre1" or entries[-1].get() == "semestre2"):
        moyenne_ue4 = calcul(lst_notes, coefficients_ue4)
        moyenne_ue5 = calcul(lst_notes, coefficients_ue5)

    # Affichage des résultats dans la console
    print(f"Moyenne UE1 : {moyenne_ue1:.2f}/20")
    print(f"Moyenne UE2 : {moyenne_ue2:.2f}/20")
    print(f"Moyenne UE3 : {moyenne_ue3:.2f}/20")
    if not (entries[-1].get() == "semestre1" or entries[-1].get() == "semestre2"):
        print(f"Moyenne UE4 : {moyenne_ue3:.2f}/20")
        print(f"Moyenne UE5 : {moyenne_ue3:.2f}/20")
    if not (entries[-1].get() == "semestre1" or entries[-1].get() == "semestre2"):
        moyennes = [moyenne_ue1, moyenne_ue2, moyenne_ue3, moyenne_ue4, moyenne_ue5]
    else:
        moyennes = [moyenne_ue1, moyenne_ue2, moyenne_ue3]
    jury(moyennes)
    
    # Afficher le graphique
    plot_graph(moyennes, bol)

btn = ttk.Button(frm, text="Valider", command=valider)
btn.grid(column=0, row=len(res)+1, columnspan=2, pady=10)

fenetre.mainloop()